# Sample images for an annotation study

Run the form cell, set your parameters, then run the Python cells below in order. The sampling unit is one row of `images.csv`: a video cover, a gallery image or an extracted frame. Each study item is an `Image ID`. Coders see the picture itself, so the codebook questions apply to what is visible.

The notebook checks that every image file exists inside `/corpus/raw` or `/corpus/derived`. The Annotation Studies service refuses anything else, and it needs the approved `images.csv` snapshot. Rows without a readable file are dropped and counted, never sampled.

Images from one post tend to look alike. Use **Max images per post** to keep a single post from dominating a sample, and **Stratified** sampling to get the same number of covers, gallery images and frames. Text and comment samples are drawn in `07_select_annotation_items`.

Keep the thumbnails private. They stay in this workspace; clear the notebook outputs before you share or commit it.

In [ ]:
import base64
import io
import json
import os
import random
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

tables_dir = Path(os.environ.get("SEMINAR_OUTPUT", "/corpus/derived")) / "tables"
media_roots = [Path("/corpus/raw"), Path("/corpus/derived")]
id_column, type_column = "Image ID", "Image Type"
style = {"description_width": "initial"}

selection_fields = {
    "item_type": widgets.Dropdown(
        options=[("Any", ""), ("Cover", "Cover"), ("Gallery", "Gallery"), ("Frame", "Frame")],
        description="Image type", style=style,
    ),
    "frame_strategy": widgets.Text(description="Frame strategy", placeholder="Any (frames only)", style=style),
    "label": widgets.Text(description="Collection label", placeholder="Any", style=style),
    "strategy": widgets.Dropdown(
        options=[("Any", ""), ("FYP", "FYP"), ("search", "search"),
                 ("hashtag", "hashtag"), ("profile", "profile")],
        description="Collection strategy", style=style,
    ),
    "max_per_post": widgets.IntText(value=0, description="Max images per post (0 = no limit)", style=style),
    "include": widgets.Textarea(description="Include only these IDs", placeholder="One ID per line", rows=2, style=style),
    "exclude": widgets.Textarea(description="Exclude these IDs", placeholder="One ID per line", rows=2, style=style),
    "sampling": widgets.Dropdown(
        options=[("All matching images", "all"), ("Simple random sample", "random"),
                 ("Stratified: same number per image type", "stratified")],
        value="random", description="Sampling", style=style,
    ),
    "size": widgets.IntText(value=20, description="Sample size (per type if stratified)", style=style),
    "seed": widgets.IntText(value=42, description="Seed", style=style),
}
title = widgets.Text(description="Working title", style=style)
filename = widgets.Text(value="image-study-draft.json", description="Draft filename", style=style)
preview_parameters = None
display(widgets.VBox([title, *selection_fields.values(), filename]))

## 1. Load the images table

Read the current widget values into an ordinary Python dictionary. IDs load as strings. Every `Path` is resolved and must point to an existing file under one of the media roots. Rows that fail are listed by reason so you can tell an instructor what is missing.

In [ ]:
preview_parameters = None
eligible = None
parameters = {name: field.value for name, field in selection_fields.items()}
source_path = tables_dir / "images.csv"

if not source_path.is_file():
    raise FileNotFoundError(f"{source_path} is unavailable. Ask your instructor for the images table.")
items = pd.read_csv(source_path, dtype=str, keep_default_na=False)
for column in (id_column, type_column, "Post ID", "Path"):
    if column not in items:
        raise ValueError(f"{source_path.name} needs a {column} column.")
if items[id_column].str.strip().eq("").any() or items[id_column].duplicated().any():
    raise ValueError(f"{source_path.name} must contain one row per non-empty {id_column}.")


def usable(path_text):
    """Return None for a readable image inside a media root, otherwise the reason it is unusable."""
    if not path_text.strip():
        return "no path"
    try:
        path = Path(path_text).resolve(strict=True)
    except OSError:
        return "file missing"
    if not path.is_file():
        return "file missing"
    if not any(path.is_relative_to(root.resolve()) for root in media_roots):
        return "outside corpus"
    return None


problems = items["Path"].map(usable)
if problems.notna().any():
    print("Dropped rows without a usable image file:")
    print(problems.dropna().value_counts().to_string())
items = items[problems.isna()].reset_index(drop=True)

print(f"Loaded {len(items)} usable images.")
display(items.groupby(type_column).size().rename("Images").reset_index())
display(items.head())

## 2. Define the eligible population

Collection filters select posts by matching label and strategy on the same membership row. Frame strategy applies to extracted frames only. Add any project-specific pandas filters here, before drawing the sample. The cap on images per post is applied in the next step, so that it uses your seed.

In [ ]:
preview_parameters = None
eligible = items.copy()

if parameters["label"].strip() or parameters["strategy"]:
    memberships = pd.read_csv(tables_dir / "collections.csv", dtype=str, keep_default_na=False)
    if parameters["label"].strip():
        memberships = memberships[memberships["label"] == parameters["label"].strip()]
    if parameters["strategy"]:
        memberships = memberships[memberships["strategy"] == parameters["strategy"]]
    eligible = eligible[eligible["Post ID"].isin(memberships["Post ID"])]

if parameters["item_type"]:
    eligible = eligible[eligible[type_column] == parameters["item_type"]]

if parameters["frame_strategy"].strip():
    if "Strategy" not in eligible:
        raise ValueError("images.csv has no Strategy column.")
    eligible = eligible[(eligible[type_column] != "Frame") | (eligible["Strategy"] == parameters["frame_strategy"].strip())]
    if parameters["item_type"] == "":
        print("Frame strategy limits frames only; covers and gallery images stay eligible.")

# Add project-specific pandas filters here.

include_ids = [line.strip() for line in parameters["include"].splitlines() if line.strip()]
exclude_ids = [line.strip() for line in parameters["exclude"].splitlines() if line.strip()]
if include_ids:
    if len(include_ids) != len(set(include_ids)):
        raise ValueError("Enter each included ID only once.")
    missing = set(include_ids).difference(eligible[id_column])
    if missing:
        raise ValueError(f"These included IDs are missing from the filtered rows: {sorted(missing)[:8]}")
    eligible = eligible.set_index(id_column, drop=False).loc[include_ids]
eligible = eligible[~eligible[id_column].isin(exclude_ids)]

print(f"{len(eligible)} eligible images after filtering.")

## 3. Draw and inspect the sample

Sorting the IDs before every draw makes the result reproducible for the same seed and eligible IDs, even if the CSV row order changes. **Max images per post** first keeps a seeded random subset of each post's images. **Simple random** then draws distinct images from what remains. **Stratified** draws the same number from every image type that remains, and stops with an error if a type has too few images. **All matching images** keeps the eligible rows in order and ignores the sample size.

In [ ]:
preview_parameters = None
if eligible is None:
    raise ValueError("Run the filter cell before drawing a sample.")
if eligible.empty:
    raise ValueError("No images match these filters.")

rng = random.Random(parameters["seed"])
population = eligible
cap = parameters["max_per_post"]
if cap < 0:
    raise ValueError("Max images per post cannot be negative.")
if cap:
    kept = []
    for post_id, group in population.groupby("Post ID", sort=True):
        ids = sorted(group[id_column])
        kept.extend(rng.sample(ids, min(cap, len(ids))))
    population = population[population[id_column].isin(kept)]
    print(f"Post limit kept {len(population)} of {len(eligible)} images.")

if parameters["sampling"] == "random":
    sample_size = parameters["size"]
    if not 1 <= sample_size <= len(population):
        raise ValueError(f"Choose a positive sample size of at most {len(population)} eligible images.")
    sampled_ids = rng.sample(sorted(population[id_column]), sample_size)
    selected = population.set_index(id_column, drop=False).loc[sampled_ids].reset_index(drop=True)
    print(f"Random sample of {len(selected)} of {len(population)} eligible images (seed {parameters['seed']}).")
elif parameters["sampling"] == "stratified":
    per_type = parameters["size"]
    sampled_ids = []
    for image_type, group in population.groupby(type_column, sort=True):
        if not 1 <= per_type <= len(group):
            raise ValueError(f"{image_type}: choose a positive number of at most {len(group)} images per type.")
        sampled_ids.extend(rng.sample(sorted(group[id_column]), per_type))
    selected = population.set_index(id_column, drop=False).loc[sampled_ids].reset_index(drop=True)
    print(f"Stratified sample of {per_type} images for each of {selected[type_column].nunique()} image types (seed {parameters['seed']}).")
else:
    selected = population.reset_index(drop=True)
    print(f"Selected all {len(selected)} eligible images.")

preview_parameters = parameters.copy()
preview_ids = selected[id_column].tolist()
display(selected.groupby(type_column).size().rename("Images").reset_index())
display(selected.head(25))

## 4. Look at the images

Check a page of thumbnails before you export. Change **Start at** to page through the sample. Thumbnails exist only in this notebook and are never part of the draft.

In [ ]:
from PIL import Image

page_size = 24
start = widgets.BoundedIntText(value=0, min=0, max=max(len(selected) - 1, 0), step=page_size, description="Start at", style=style)
gallery_output = widgets.Output()


def thumbnail(path):
    with Image.open(path) as image:
        image = image.convert("RGB")
        image.thumbnail((180, 180))
        buffer = io.BytesIO()
        image.save(buffer, "JPEG", quality=80)
    return base64.b64encode(buffer.getvalue()).decode("ascii")


def show_page(_=None):
    rows = selected.iloc[start.value:start.value + page_size]
    cards = []
    for _, row in rows.iterrows():
        try:
            picture = f'<img src="data:image/jpeg;base64,{thumbnail(row["Path"])}" style="max-width:180px;max-height:180px">'
        except Exception as error:
            picture = f"<em>unreadable: {type(error).__name__}</em>"
        cards.append(f'<div style="display:inline-block;margin:4px;vertical-align:top;font-size:11px">{picture}<br>{row[id_column]}<br>{row[type_column]}</div>')
    gallery_output.clear_output()
    with gallery_output:
        print(f"Images {start.value + 1} to {start.value + len(rows)} of {len(selected)}.")
        display(widgets.HTML("".join(cards)))


start.observe(show_page, names="value")
display(start, gallery_output)
show_page()

## 5. Export the reviewed draft

Once the thumbnails look right, run this cell. If you change any selection inputs in the form, rerun the load, filter and sample cells before exporting. The JSON contains the title, source table and ordered IDs, never file paths. Download it from `annotation-drafts` in the JupyterLab file browser. Open **Services → Annotation Studies → Create study**, upload the JSON under **Notebook draft**, write your instructions and questions, and click **Preview coding form**. Check the images and their order before you click **Publish to class**.

In [ ]:
if preview_parameters is None:
    raise ValueError("Run the load, filter, and sample cells before exporting.")
if preview_parameters != {name: field.value for name, field in selection_fields.items()}:
    raise ValueError("Selection inputs changed. Rerun the load, filter, and sample cells before exporting.")

working_title = title.value.strip()
if not 1 <= len(working_title) <= 200:
    raise ValueError("Enter a working title of up to 200 characters.")
draft_name = filename.value.strip()
if Path(draft_name).name != draft_name or Path(draft_name).suffix != ".json":
    raise ValueError("Enter a draft filename ending in .json.")

draft_dir = Path.home() / "annotation-drafts"
draft_dir.mkdir(mode=0o700, parents=True, exist_ok=True)
draft_path = draft_dir / draft_name
draft = {
    "version": 1,
    "title": working_title,
    "source": "images",
    "item_ids": preview_ids,
}

# Exclusive creation protects any draft saved under the same name.
with draft_path.open("x", encoding="utf-8") as output:
    json.dump(draft, output, ensure_ascii=False, indent=2)
    output.write("\n")
print(f"Saved {len(draft['item_ids'])} image IDs to {draft_path}.")